# LoomHash Inference module -- Stage 1 (distillation) training

Trains the ONNX regressor described in `docs/open-decisions.md` D-04: a MobileNetV4 backbone + linear head that maps a 224x224 RGB face image directly to the 128-d vector, replacing MediaPipe + `edge/feature_extraction.mjs`'s geometric math at inference time.

**Training strategy (D-13-adjacent, see STATUS.md):** this notebook does **Stage 1 only** -- distillation. It runs MediaPipe + the existing, already-verified `training/feature_targets.py` (a byte-for-byte Python port of `edge/feature_extraction.mjs`, cross-validated against the real JS implementation) on FairFace images to generate target vectors, then trains the CNN to reproduce that output directly from pixels. Stage 2 (metric learning / triplet loss, for actual verification accuracy) needs an identity-labeled dataset with confirmed commercial rights, which hasn't been sourced yet -- see the dataset-licensing findings in STATUS.md/chat history. Do not add Stage 2 code here until that's resolved.

**Dataset: FairFace** (CC BY 4.0, commercial use permitted with attribution to Karkkainen & Joo -- verified against the FairFace GitHub repo before use, since two of the four originally-proposed datasets, CelebA and WIDER FACE, turned out to be explicitly non-commercial-only and were dropped). No identity labels -- FairFace only supports Stage 1.

**Before running:** Runtime > Change runtime type > select a GPU (T4 is plenty for this model size).

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU. Runtime > Change runtime type > select a GPU (e.g. T4), then re-run.')

## 1. Setup: clone the repo, install training-only deps

Clones the actual LoomHash repo so this notebook trains against the exact same `feature_targets.py` that's committed and tested there -- not a copy-pasted duplicate that could silently drift out of sync.

In [ ]:
!pip install -q mediapipe timm datasets onnx onnxruntime
!git clone -q https://github.com/eddy7896/loomhash.git
%cd loomhash

import sys
sys.path.insert(0, 'training')
from feature_targets import NUM_LANDMARKS, VECTOR_DIM, FEATURE_VERSION, extract_feature_vector
print('Using feature_targets.py FEATURE_VERSION =', FEATURE_VERSION)

## 2. Download FairFace

Start with a subset to validate the pipeline runs correctly end to end before committing a full session to the entire ~86K-image train split -- increase `FAIRFACE_SUBSET_SIZE` (or set to `None` for the full split) once you've confirmed this works.

In [ ]:
from datasets import load_dataset

FAIRFACE_SUBSET_SIZE = 20000  # None = full train split (~86,744 images)

print('Downloading FairFace...')
fairface = load_dataset('HuggingFaceM4/FairFace', split='train')
print(f'Full train split: {len(fairface)} images')

if FAIRFACE_SUBSET_SIZE is not None:
    fairface = fairface.select(range(min(FAIRFACE_SUBSET_SIZE, len(fairface))))
print(f'Using {len(fairface)} images for this run')

## 3. Download the MediaPipe FaceLandmarker model (for offline label generation only)

This is a training-time data-prep step, not the production server -- does not violate system.md constraint 4 (see `training/generate_labels.py`'s docstring).

In [ ]:
import os
os.makedirs('training/models', exist_ok=True)
!curl -sL -o training/models/face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
print('Model downloaded:', os.path.getsize('training/models/face_landmarker.task'), 'bytes')

## 4. Generate distillation labels

Runs MediaPipe on each FairFace image (in memory, no disk round-trip) and computes the target vector via the verified `extract_feature_vector`. Images where MediaPipe detects no face are skipped -- expect a nonzero no-face rate; FairFace isn't curated for face-detection success.

In [ ]:
import numpy as np
import mediapipe as mp
from mediapipe.tasks.python import BaseOptions, vision

options = vision.FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='training/models/face_landmarker.task'),
    running_mode=vision.RunningMode.IMAGE,
    num_faces=1,
)
landmarker = vision.FaceLandmarker.create_from_options(options)

def pil_to_mp_image(pil_image):
    rgb = pil_image.convert('RGB')
    arr = np.asarray(rgb, dtype=np.uint8)
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=arr), rgb

images = []
targets = []
skipped_no_face = 0
skipped_error = 0

for i, example in enumerate(fairface):
    if i % 1000 == 0:
        print(f'[{i}/{len(fairface)}] kept={len(images)} no_face={skipped_no_face} error={skipped_error}')
    try:
        mp_image, rgb_image = pil_to_mp_image(example['image'])
        result = landmarker.detect(mp_image)
        if not result.face_landmarks:
            skipped_no_face += 1
            continue
        points = result.face_landmarks[0][:NUM_LANDMARKS]
        landmarks = np.array([[p.x, p.y, p.z] for p in points], dtype=np.float64)
        target_vector = extract_feature_vector(landmarks)
        images.append(rgb_image)  # kept at native size; resized per-item during training
        targets.append(target_vector.astype(np.float32))
    except Exception as exc:
        skipped_error += 1

print(f'Done. kept={len(images)} no_face={skipped_no_face} error={skipped_error}')
targets = np.stack(targets)
print('targets shape:', targets.shape)

assert len(images) > 100, 'Too few labeled images to train on -- check the no_face/error counts above.'

## 5. Dataset / DataLoader

Normalization matches the agreed ONNX I/O contract exactly (D-04): 224x224, ImageNet mean/std.

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class DistillationDataset(Dataset):
    def __init__(self, images, targets, transform):
        self.images = images
        self.targets = targets
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.transform(self.images[idx])
        target = torch.from_numpy(self.targets[idx])
        return img, target

n_val = max(1, int(0.1 * len(images)))
train_images, val_images = images[:-n_val], images[-n_val:]
train_targets, val_targets = targets[:-n_val], targets[-n_val:]

train_ds = DistillationDataset(train_images, train_targets, transform)
val_ds = DistillationDataset(val_images, val_targets, transform)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'train: {len(train_ds)}  val: {len(val_ds)}')

## 6. Model: MobileNetV4 backbone + linear regression head

Per the agreed architecture (D-04): MobileNetV4 (via `timm` -- not in torchvision) as a feature backbone, no classification head, followed by a single linear layer to 128 dimensions.

In [ ]:
import timm
import torch.nn as nn

class LoomEngine(nn.Module):
    def __init__(self, backbone_name='mobilenetv4_conv_small', vector_dim=128, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        self.head = nn.Linear(self.backbone.num_features, vector_dim)

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

device = 'cuda'
model = LoomEngine().to(device)
print(f'backbone features: {model.backbone.num_features} -> head -> {VECTOR_DIM}')
print('total params:', sum(p.numel() for p in model.parameters()))

## 7. Train (MSE regression to the distillation targets)

In [ ]:
import torch.optim as optim

EPOCHS = 15
LR = 1e-3

optimizer = optim.AdamW(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
loss_fn = nn.MSELoss()

best_val_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for imgs, tgt in train_loader:
        imgs, tgt = imgs.to(device), tgt.to(device)
        optimizer.zero_grad()
        pred = model(imgs)
        loss = loss_fn(pred, tgt)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, tgt in val_loader:
            imgs, tgt = imgs.to(device), tgt.to(device)
            pred = model(imgs)
            val_loss += loss_fn(pred, tgt).item() * imgs.size(0)
    val_loss /= len(val_ds)
    scheduler.step()

    print(f'epoch {epoch+1}/{EPOCHS}  train_mse={train_loss:.5f}  val_mse={val_loss:.5f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'loom_engine_best.pt')
        print('  saved new best checkpoint')

print('Training done. Best val MSE:', best_val_loss)

## 8. Export to ONNX (fp32), matching the agreed I/O contract exactly

In [ ]:
model.load_state_dict(torch.load('loom_engine_best.pt', map_location=device))
model.eval()

dummy_input = torch.randn(1, 3, 224, 224, device=device)

torch.onnx.export(
    model,
    dummy_input,
    'loom_engine_fp32.onnx',
    input_names=['input'],
    output_names=['output'],
    dynamic_axes=None,  # fixed batch size 1, per the agreed I/O contract (D-04)
    opset_version=17,
)
print('Exported loom_engine_fp32.onnx')

import onnx
onnx_model = onnx.load('loom_engine_fp32.onnx')
onnx.checker.check_model(onnx_model)
print('ONNX model is structurally valid.')
in_shape = [d.dim_value for d in onnx_model.graph.input[0].type.tensor_type.shape.dim]
out_shape = [d.dim_value for d in onnx_model.graph.output[0].type.tensor_type.shape.dim]
print('input:', onnx_model.graph.input[0].name, in_shape)
print('output:', onnx_model.graph.output[0].name, out_shape)
assert in_shape == [1, 3, 224, 224], f'input shape mismatch: {in_shape}'
assert out_shape == [1, 128], f'output shape mismatch: {out_shape}'

## 9. Static INT8 quantization

Per the original spec's math section C. Uses a slice of held-out validation images as calibration data.

In [ ]:
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat

class FairFaceCalibrationReader(CalibrationDataReader):
    def __init__(self, images, transform, n=200):
        self.images = images[:n]
        self.transform = transform
        self.idx = 0

    def get_next(self):
        if self.idx >= len(self.images):
            return None
        tensor = self.transform(self.images[self.idx]).unsqueeze(0).numpy().astype(np.float32)
        self.idx += 1
        return {'input': tensor}

calibration_reader = FairFaceCalibrationReader(val_images, transform, n=min(200, len(val_images)))

quantize_static(
    model_input='loom_engine_fp32.onnx',
    model_output='loom_engine_int8.onnx',
    calibration_data_reader=calibration_reader,
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QInt8,
    weight_type=QuantType.QInt8,
)

fp32_size = os.path.getsize('loom_engine_fp32.onnx') / 1e6
int8_size = os.path.getsize('loom_engine_int8.onnx') / 1e6
print(f'fp32: {fp32_size:.2f} MB   int8: {int8_size:.2f} MB')

## 10. Verify the quantized model

Checks shape/load correctness and compares int8 vs fp32 output on a real image (cosine similarity) as a sanity check that quantization didn't destroy the model -- this is NOT a biometric accuracy claim; that requires the separate consented evaluation protocol in `docs/open-decisions.md`.

In [ ]:
import onnxruntime as ort
import time

session = ort.InferenceSession('loom_engine_int8.onnx', providers=['CPUExecutionProvider'])
test_input = np.random.randn(1, 3, 224, 224).astype(np.float32)

start = time.time()
outputs = session.run(['output'], {'input': test_input})
elapsed_ms = (time.time() - start) * 1000
print('output shape:', outputs[0].shape)
print('inference time (first call, includes warmup):', round(elapsed_ms, 2), 'ms')

fp32_session = ort.InferenceSession('loom_engine_fp32.onnx', providers=['CPUExecutionProvider'])
real_input = transform(val_images[0]).unsqueeze(0).numpy().astype(np.float32)
fp32_out = fp32_session.run(['output'], {'input': real_input})[0]
int8_out = session.run(['output'], {'input': real_input})[0]
cos_sim = float(np.dot(fp32_out[0], int8_out[0]) / (np.linalg.norm(fp32_out[0]) * np.linalg.norm(int8_out[0])))
print('fp32 vs int8 cosine similarity on a real image:', round(cos_sim, 4))

## 11. Download the trained model

After downloading, copy `loom_engine_int8.onnx` into the LoomHash repo (e.g. `loomhash/inference/models/`, once that module exists per D-04/D-12) and update `docs/verification-checklist.md` and `docs/open-decisions.md` to record that a real trained artifact now exists -- everything in this repo has referred to it as "does not exist yet" up to this point.

In [ ]:
from google.colab import files
files.download('loom_engine_int8.onnx')